In [1]:
import pandas as pd
import numpy as np
import os
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [6]:
data_path = "dataset/"
files = {}
for f in os.listdir(data_path):
    name = f.replace(".csv", "")
    files[name] = pd.read_csv(data_path + f)
    print(f"{f}: {files[name].shape}")

web_traffic.csv: (3652, 7)
customers.csv: (121930, 7)
products.csv: (2412, 8)
reviews.csv: (113551, 7)
orders.csv: (646945, 8)
shipments.csv: (566067, 4)
promotions.csv: (50, 10)
geography.csv: (39948, 4)
payments.csv: (646945, 4)
order_items.csv: (714669, 7)
inventory.csv: (60247, 17)
sample_submission.csv: (548, 3)
returns.csv: (39939, 7)
sales.csv: (3833, 3)


/var/folders/f3/db87bv813793dtqfxfgthbcr0000gn/T/ipykernel_50219/486795206.py:5: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  files[name] = pd.read_csv(data_path + f)


In [38]:
files['sales']['gross_margin'] = (files['sales']['Revenue'] - files['sales']['COGS']) / files['sales']['Revenue']
files['sales']['Date'] = pd.to_datetime(files['sales']['Date'])

In [49]:
files['promotions']['start_date'] = pd.to_datetime(files['promotions']['start_date'])
files['promotions']['end_date'] = pd.to_datetime(files['promotions']['end_date'])
files['promotions']['duration'] = (files['promotions']['end_date'] - files['promotions']['start_date']).dt.total_seconds() * 1000

In [53]:
fig = make_subplots()

fig.add_trace(go.Scatter(x=files['sales']['Date'], y=files['sales']['Revenue'], mode='lines'))
fig.add_trace(go.Scatter(x=files['sales']['Date'], y=files['sales']['gross_margin'], mode='lines', opacity=0.4, yaxis='y2'))
fig.add_trace(go.Bar(x=files['promotions']['start_date'], y=files['promotions']['discount_value'], width=files['promotions']['duration'], offset=0, opacity=0.4, yaxis='y3', marker_color='purple'))


fig.update_xaxes(
    tickformat='%b %Y',
    dtick='M1',
    tickfont=dict(size=7)
)

fig.update_layout(
    width = 1300,

    yaxis2=dict(
        title="Gross Margin",
        overlaying='y',
        side='right',
        showgrid=False,
        tickfont=dict(size=7)
    ),

    yaxis3=dict(
        title="Promotions",
        overlaying='y',
        side='right',
        showgrid=False,
        tickfont=dict(size=7),
        anchor="free",
        position=0.98
    )
)
fig.show()

In [54]:
daily = files['sales'].copy()

In [56]:
def get_discount(current_date):
    # Find rows where the date is within the promotion range
    mask = (files['promotions']['start_date'] <= current_date) & (current_date <= files['promotions']['end_date'])
    valid_promos = files['promotions'].loc[mask]

    if not valid_promos.empty:
        # Return the max discount if multiple promotions overlap
        return valid_promos['discount_value'].max()
    return 0

In [57]:
daily['promotions_discount_value'] = daily['Date'].apply(get_discount)

In [66]:
files['web_traffic']['date'] = pd.to_datetime(files['web_traffic']['date'])
daily = daily.merge(files['web_traffic'][['date', 'sessions', 'traffic_source']], left_on='Date', right_on='date', how='left')

In [71]:
daily

,Date,Revenue,COGS,gross_margin,promotions_discount_value,sessions,traffic_source
0,2012-07-04,5123547.94,3982991.19,0.222611,0.0,NaN,NaN
1,2012-07-05,2751773.45,2150580.23,0.218475,0.0,NaN,NaN
2,2012-07-06,3054029.42,2517632.84,0.175636,0.0,NaN,NaN
3,2012-07-07,2667930.94,2108246.62,0.209782,0.0,NaN,NaN
4,2012-07-08,2360851.90,1808622.79,0.233911,0.0,NaN,NaN
...,...,...,...,...,...,...,...
3828,2022-12-27,2100553.66,2184872.24,-0.040141,20.0,17416.0,organic_search
3829,2022-12-28,3448729.20,3513621.00,-0.018816,20.0,21071.0,organic_search
3830,2022-12-29,3083944.33,3170787.10,-0.028160,20.0,20884.0,direct
3831,2022-12-30,2884668.76,3022292.15,-0.047709,20.0,17679.0,email_campaign


In [72]:
daily['sessions'] = daily['sessions'].fillna(daily['sessions'].mean())
daily['traffic_source'] = daily['traffic_source'].fillna('Unknown')

In [74]:
temp = daily.copy()
temp.groupby('traffic_source')['sessions'].sum().reset_index().sort_values('sessions', ascending=False)

,traffic_source,sessions
3,organic_search,2.719698e+07
4,paid_search,1.959827e+07
6,social_media,1.581623e+07
2,email_campaign,1.279267e+07
5,referral,9.476845e+06
1,direct,6.571549e+06
0,Unknown,4.532560e+06


In [80]:
px.histogram(temp, x='traffic_source', y='sessions', color='traffic_source')